# Phase 1 CME Reference Demo

This notebook is the required end-to-end Phase 1 showcase. It should be fully executed before Phase 1 is considered complete.

Expected reference inputs:

- `data/input/Metals_Option_Products.pdf`
- `data/input/metal_options_summary.rtf`

The implementation must stay generic. The CME bulletin is a difficult reference case, not a hard-coded document type.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
PDF_PATH = PROJECT_ROOT / "data" / "input" / "Metals_Option_Products.pdf"
SUMMARY_PATH = PROJECT_ROOT / "data" / "input" / "metal_options_summary.rtf"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / "notebook_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF exists:", PDF_PATH.exists(), PDF_PATH)
print("Summary exists:", SUMMARY_PATH.exists(), SUMMARY_PATH)
assert PDF_PATH.exists(), f"Missing reference PDF: {PDF_PATH}"
assert SUMMARY_PATH.exists(), f"Missing reference summary: {SUMMARY_PATH}"

## 1. Import the Phase 1 package

These imports should come from reusable source code, not notebook-only helper implementations.

In [ ]:
from large_pdf_extractor.app.pipeline import Phase1Pipeline
from large_pdf_extractor.app.config import RunConfig
from large_pdf_extractor.domain.models import ParseStrategy, ExtractionDictionary, ExtractionResult, ComparisonReport

print("Phase 1 package imported successfully")

## 2. Propose the dictionary first

The pipeline must build or load the extraction dictionary before final extraction.

In [ ]:
config = RunConfig(
    pdf_path=str(PDF_PATH),
    output_dir=str(OUTPUT_DIR),
    strategy=ParseStrategy.PYMUPDF,
    llm_provider="fake",
    max_chunk_tokens=4000,
    chunk_overlap_tokens=300,
    max_chunks=12,
)

pipeline = Phase1Pipeline(config)
dictionary = pipeline.propose_dictionary()

assert isinstance(dictionary, ExtractionDictionary)
assert len(dictionary.items) > 0
print("Dictionary items:", len(dictionary.items))
for item in dictionary.items[:5]:
    print(f"- {item.item_id}: {item.entity_name} [{item.expected_type}]")

## 3. Run PyMuPDF path

This is the required fast baseline path.

In [ ]:
pymupdf_result = pipeline.run(strategy=ParseStrategy.PYMUPDF, dictionary=dictionary)

assert pymupdf_result.extraction_result is not None
print("PyMuPDF chunks:", len(pymupdf_result.chunks))
print("PyMuPDF extracted values:", len(pymupdf_result.extraction_result.values))

## 4. Run Docling path or graceful skip

Docling should run when installed. If unavailable, the result must record a structured skip/warning instead of failing the notebook.

In [ ]:
docling_result = pipeline.run(strategy=ParseStrategy.DOCLING, dictionary=dictionary, allow_parser_skip=True)

print("Docling warnings:", docling_result.warnings)
print("Docling errors:", docling_result.errors)
if docling_result.extraction_result:
    print("Docling chunks:", len(docling_result.chunks))
    print("Docling extracted values:", len(docling_result.extraction_result.values))
else:
    print("Docling path skipped or unavailable; this is acceptable only if captured in the result.")

## 5. Run compare mode

Compare mode must use the same dictionary for both parser/chunking paths.

In [ ]:
compare_result = pipeline.run(strategy=ParseStrategy.COMPARE, dictionary=dictionary, allow_parser_skip=True)
report = compare_result.comparison_report

assert isinstance(report, ComparisonReport)
print("Compared strategies:", report.compared_strategies)
print("Comparison warnings:", report.warnings)

## 6. Validate generated artifacts

Core JSON and Markdown artifacts should exist and should reload into Pydantic models.

In [ ]:
expected_any = [
    "extraction_dictionary.proposed.json",
    "extraction_dictionary.used.json",
    "comparison_report.json",
    "comparison_report.md",
]

existing = sorted(p.relative_to(OUTPUT_DIR) for p in OUTPUT_DIR.rglob("*") if p.is_file())
print("Generated artifact count:", len(existing))
for path in existing[:50]:
    print("-", path)

# Validate at least one extraction JSON exists.
extraction_jsons = list(OUTPUT_DIR.rglob("extraction_result*.json"))
assert extraction_jsons, "No extraction_result JSON artifacts found"

with extraction_jsons[0].open() as f:
    loaded_result = ExtractionResult.model_validate_json(f.read())
print("Validated extraction result:", extraction_jsons[0].name, len(loaded_result.values), "values")

## 7. Markdown previews

In [ ]:
for md_path in list(OUTPUT_DIR.rglob("*.md"))[:3]:
    print("\n---", md_path.relative_to(OUTPUT_DIR), "---")
    text = md_path.read_text(encoding="utf-8")
    print(text[:2500])

## 8. Acceptance checklist

The final cell should print pass/fail status for the Phase 1 notebook demonstration.

In [ ]:
checks = {
    "input_pdf_exists": PDF_PATH.exists(),
    "summary_exists": SUMMARY_PATH.exists(),
    "dictionary_has_items": len(dictionary.items) > 0,
    "pymupdf_extraction_created": pymupdf_result.extraction_result is not None,
    "comparison_report_created": report is not None,
    "extraction_json_exists": bool(extraction_jsons),
}

for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(checks.values()), checks
print("\nPhase 1 notebook demo completed successfully.")